In [20]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold

raw = pd.read_csv("../data/raw_chembl_tid1.csv")
n_raw = len(raw)

In [21]:
human_mask = (raw["assay_tax_id"] == 9606) | (raw["assay_organism"] == "Homo sapiens")
n_nonhuman_excluded = int((~human_mask).sum())
df = raw[human_mask].copy()

In [22]:
ALLOWED_TYPES = {"IC50", "Ki", "Kd", "EC50"}
type_mask = df["activity_type"].isin(ALLOWED_TYPES)
n_assaytype_excluded = int((~type_mask).sum())
df = df[type_mask].copy()

In [23]:
conf_mask = df["confidence_score"].fillna(0) >= 7
n_confidence_excluded = int((~conf_mask).sum())
df = df[conf_mask].copy()

In [24]:
UNIT_FACTORS = {"nM": 1.0, "uM": 1e3, "µM": 1e3, "mM": 1e6, "M": 1e9, "pM": 1e-3}

def to_nM(row):
    f = UNIT_FACTORS.get(row["standard_units_raw"])
    if f is None or pd.isna(row["standard_value_raw"]):
        return None
    return float(row["standard_value_raw"]) * f

df["standard_value_nM"] = df.apply(to_nM, axis=1)
n_unit_excluded = int(df["standard_value_nM"].isna().sum())
df = df[df["standard_value_nM"].notna()].copy()

In [25]:
_lc = rdMolStandardize.LargestFragmentChooser()
_un = rdMolStandardize.Uncharger()

def standardize(smiles):
    """Return (canonical_smiles, inchikey, n_undefined_stereo) or None on failure."""
    if pd.isna(smiles):
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = _lc.choose(mol)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    mol = _un.uncharge(mol)
    can = Chem.MolToSmiles(mol, canonical=True)
    key = Chem.MolToInchiKey(mol)
    if not key:
        return None
    centers = Chem.FindMolChiralCenters(mol, includeUnassigned=True,
                                        useLegacyImplementation=False)
    n_undef = sum(1 for _, c in centers if c == "?")
    return can, key, n_undef

std = df["canonical_smiles"].apply(standardize)
df["std_smiles"]   = std.apply(lambda x: x[0] if x else None)
df["std_inchikey"] = std.apply(lambda x: x[1] if x else None)
df["undefined_stereocenters"] = std.apply(lambda x: x[2] if x else 0)
n_standardization_failed = int(df["std_inchikey"].isna().sum())
df = df[df["std_inchikey"].notna()].copy()

[08:50:47] Running LargestFragmentChooser
[08:50:47] Running Uncharger
[08:50:47] Running LargestFragmentChooser
[08:50:47] Running Uncharger
[08:50:47] Running LargestFragmentChooser
[08:50:47] Running Uncharger
[08:50:47] Running LargestFragmentChooser
[08:50:47] Running Uncharger
[08:50:47] Running LargestFragmentChooser
[08:50:47] Running Uncharger
[08:50:47] Running LargestFragmentChooser
[08:50:47] Running Uncharger
[08:50:47] Running LargestFragmentChooser
[08:50:47] Fragment: OC[C@@H]1[C@@H](O)[C@H](O)C[S@@+]1C[C@@H](O)[C@@H](O)[C@H](O)[C@@H](O)CO
[08:50:47] New largest fragment: OC[C@@H]1[C@@H](O)[C@H](O)C[S@@+]1C[C@@H](O)[C@@H](O)[C@H](O)[C@@H](O)CO (43)
[08:50:47] Fragment: [Cl-]
[08:50:47] Running Uncharger
[08:50:47] Running LargestFragmentChooser
[08:50:47] Fragment: OC[C@@H](O)[C@@H](O)[C@H](O)[C@H](O)C[S@+]1C[C@@H](O)[C@H](O)[C@H]1CO
[08:50:47] New largest fragment: OC[C@@H](O)[C@@H](O)[C@H](O)[C@H](O)C[S@+]1C[C@@H](O)[C@H](O)[C@H]1CO (43)
[08:50:47] Fragment: [Cl-]
[08

In [26]:
def classify(relation, value_nM):
    if value_nM is None or pd.isna(value_nM):
        return "unparseable"
    if relation in ("=", "<=", "<") and value_nM <= 1000:
        return "active"
    if relation in ("=", ">", ">=") and value_nM >= 10000:
        return "inactive"
    if 1000 < value_nM < 10000:
        return "ambiguous"
    return "unparseable"

df["activity_class"] = df.apply(
    lambda r: classify(r["standard_relation"], r["standard_value_nM"]), axis=1)
n_unparseable_records = int((df["activity_class"] == "unparseable").sum())
df = df[df["activity_class"] != "unparseable"].copy()

In [27]:
# Compound-level conflict check (per data.md §1.1.1)
by_key = df.groupby("std_inchikey")["activity_class"]
has_active   = by_key.apply(lambda s: (s == "active").any())
has_inactive = by_key.apply(lambda s: (s == "inactive").any())
conflict_keys = set(has_active[has_active & has_inactive].index)
n_activity_conflict_excluded = len(conflict_keys)
df = df[~df["std_inchikey"].isin(conflict_keys)].copy()

# Consolidation: median value across remaining records
def consolidate(g):
    med = g["standard_value_nM"].median()
    best = g.sort_values(["confidence_score", "activity_id"],
                         ascending=[False, True]).iloc[0]
    ratio = g["standard_value_nM"].max() / g["standard_value_nM"].min() \
            if g["standard_value_nM"].min() > 0 else 1.0
    return pd.Series({
        "molecule_chembl_id": best["molecule_chembl_id"],
        "parent_molecule_chembl_id": best["parent_molecule_chembl_id"],
        "target_chembl_id": best["target_chembl_id"],
        "assay_chembl_id": best["assay_chembl_id"],
        "document_chembl_id": best["document_chembl_id"],
        "activity_type": best["activity_type"],
        "standard_relation": best["standard_relation"],
        "standard_value_raw": best["standard_value_raw"],
        "standard_units_raw": best["standard_units_raw"],
        "standard_value_nM": med,
        "species": "Homo sapiens",
        "tax_id": 9606,
        "confidence_score": best["confidence_score"],
        "std_smiles": best["std_smiles"],
        "undefined_stereocenters": int(best["undefined_stereocenters"]),
        "duplicate_flag": bool(ratio > 10),
        "n_records": len(g),
    })

# The .reset_index() will now create the 'std_inchikey' column cleanly
curated = df.groupby("std_inchikey", group_keys=False).apply(consolidate, include_groups=False).reset_index()
curated["activity_class"] = curated.apply(
    lambda r: classify(r["standard_relation"], r["standard_value_nM"]), axis=1)

n_actives   = int((curated["activity_class"] == "active").sum())
n_inactives = int((curated["activity_class"] == "inactive").sum())
n_ambiguous = int((curated["activity_class"] == "ambiguous").sum())
n_duplicate_flagged = int(curated["duplicate_flag"].sum())

print(f"Compounds before consolidation: {len(df)}")
print(f"Activity conflicts excluded: {n_activity_conflict_excluded}")
print(f"Compounds after consolidation: {len(curated)}")
print(f"Actives: {n_actives}")
print(f"Inactives: {n_inactives}")
print(f"Ambiguous (1-10 uM): {n_ambiguous}")

Compounds before consolidation: 313
Activity conflicts excluded: 2
Compounds after consolidation: 287
Actives: 31
Inactives: 148
Ambiguous (1-10 uM): 108


In [28]:
curated_final = (
    curated.rename(columns={
        "std_smiles": "canonical_smiles",
        "std_inchikey": "standard_inchikey",
    })
    [[
        "molecule_chembl_id", "parent_molecule_chembl_id", "target_chembl_id",
        "assay_chembl_id", "document_chembl_id", "activity_type",
        "standard_relation", "standard_value_raw", "standard_units_raw",
        "standard_value_nM", "species", "tax_id", "confidence_score",
        "canonical_smiles", "standard_inchikey", "undefined_stereocenters",
        "activity_class", "duplicate_flag", "n_records",
    ]]
)

curated_final.to_csv("../data/curated_chembl.csv", index=False)
print(f"Curated compounds: {len(curated_final)}")
print(curated_final["activity_class"].value_counts().to_string())

Curated compounds: 287
activity_class
inactive     148
ambiguous    108
active        31


In [29]:
import json, hashlib
from pathlib import Path

SEED = 42

def scaffold_of(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ""
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)

curated_final["scaffold"] = curated_final["canonical_smiles"].apply(scaffold_of)

# Group by scaffold (indices into curated_final)
groups = {}
for i, scaf in enumerate(curated_final["scaffold"]):
    groups.setdefault(scaf, []).append(i)

sorted_groups = sorted(groups.items(), key=lambda kv: (-len(kv[1]), kv[0]))

n_total = len(curated_final)
train_target = int(0.70 * n_total)
val_target   = int(0.85 * n_total)

train_idx, val_idx, test_idx = [], [], []
for scaffold, idxs in sorted_groups:
    if len(train_idx) < train_target:
        train_idx.extend(idxs)
    elif len(train_idx) + len(val_idx) < val_target:
        val_idx.extend(idxs)
    else:
        test_idx.extend(idxs)

# Active-stratification repair (max 3)
split_repairs = []
def n_actives_in(idxs):
    return int((curated_final.iloc[idxs]["activity_class"] == "active").sum())

for _ in range(3):
    if n_actives_in(val_idx) > 0 and n_actives_in(test_idx) > 0:
        break
    if n_actives_in(test_idx) == 0:
        deficient = "test"
    elif n_actives_in(val_idx) == 0:
        deficient = "val"
    else:
        break
    # Move smallest active-containing train group of size <= 5
    candidates = [(s, idxs) for s, idxs in sorted_groups
                  if set(idxs).issubset(set(train_idx))
                  and len(idxs) <= 5
                  and n_actives_in(idxs) > 0]
    if not candidates:
        break
    candidates.sort(key=lambda kv: (len(kv[1]), kv[0]))
    scaf, idxs = candidates[0]
    if deficient == "test":
        test_idx.extend(idxs)
    else:
        val_idx.extend(idxs)
    train_idx = [i for i in train_idx if i not in idxs]
    split_repairs.append({"scaffold": scaf, "moved_to": deficient, "size": len(idxs)})

In [30]:
curated_sha = hashlib.sha256(Path("../data/curated_chembl.csv").read_bytes()).hexdigest()

split_indices = {
    "schema_version": "1.0.0",
    "seed": SEED,
    "chembl_release": 37,
    "download_date": "2026-11-03",
    "curated_csv_sha256": curated_sha,
    "scaffold_definition": "bemis_murcko_generic",
    "split_rule": "greedy_size_desc_70_15_15_with_active_repair",
    "split_repairs": split_repairs,
    "n_compounds": {"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)},
    "n_actives": {
        "train": n_actives_in(train_idx),
        "validation": n_actives_in(val_idx),
        "test": n_actives_in(test_idx),
    },
    "indices": {"train": train_idx, "validation": val_idx, "test": test_idx},
}
Path("../splits").mkdir(exist_ok=True)
Path("../splits/split_indices.json").write_text(json.dumps(split_indices, indent=2))

# CV folds within train: round-robin over scaffolds sorted by size desc
train_groups = [(s, idxs) for s, idxs in sorted_groups
                if set(idxs).issubset(set(train_idx))]
train_groups.sort(key=lambda kv: (-len(kv[1]), kv[0]))
folds = {str(i): [] for i in range(1, 6)}
for i, (scaf, idxs) in enumerate(train_groups):
    folds[str((i % 5) + 1)].extend(idxs)

scaffold_of_compound = {str(i): curated_final.iloc[i]["scaffold"]
                        for i in range(len(curated_final))}
cv = {
    "schema_version": "1.0.0",
    "seed": SEED,
    "n_folds": 5,
    "fold_assignment_rule": "size_desc_round_robin_over_train_groups",
    "scaffold_of_compound": scaffold_of_compound,
    "folds": folds,
}
Path("../splits/cv_scaffold_groups.json").write_text(json.dumps(cv, indent=2))
print("Split sizes:", split_indices["n_compounds"])
print("Split actives:", split_indices["n_actives"])
print("Repairs:", split_repairs)

Split sizes: {'train': 201, 'validation': 42, 'test': 44}
Split actives: {'train': 22, 'validation': 4, 'test': 5}
Repairs: []


In [31]:
train_keys = set(curated_final.iloc[train_idx]["standard_inchikey"])
before = {"val": len(val_idx), "test": len(test_idx)}

val_idx  = [i for i in val_idx  if curated_final.iloc[i]["standard_inchikey"] not in train_keys]
test_idx = [i for i in test_idx if curated_final.iloc[i]["standard_inchikey"] not in train_keys]

leakage_removed = (before["val"] - len(val_idx)) + (before["test"] - len(test_idx))

# rewrite split_indices with cleaned indices
split_indices["indices"] = {"train": train_idx, "validation": val_idx, "test": test_idx}
split_indices["n_compounds"] = {"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)}
split_indices["n_actives"] = {
    "train": n_actives_in(train_idx),
    "validation": n_actives_in(val_idx),
    "test": n_actives_in(test_idx),
}
Path("../splits/split_indices.json").write_text(json.dumps(split_indices, indent=2))
print("Leakage removed:", leakage_removed)

Leakage removed: 0


In [32]:
import json
import pandas as pd
from pathlib import Path

# 1. Load the split files
splits = json.loads(Path("../splits/split_indices.json").read_text())
train_idx = splits["indices"]["train"]
val_idx = splits["indices"]["validation"]
test_idx = splits["indices"]["test"]

# 2. Count scaffold groups per split
def get_scaffold_stats(idxs):
    scaffolds = [curated_final.iloc[i]["scaffold"] for i in idxs]
    counts = pd.Series(scaffolds).value_counts()
    return len(counts), counts.max() if len(counts) > 0 else 0

groups_tr, max_tr = get_scaffold_stats(train_idx)
groups_val, max_val = get_scaffold_stats(val_idx)
groups_te, max_te = get_scaffold_stats(test_idx)

# 3. Compile the ledger metrics
# Note: If you didn't save n_nonhuman_excluded, n_confidence_excluded, etc. 
# in exact variables during the earlier cells, you can calculate them from `raw`:
raw = pd.read_csv("../data/raw_chembl_tid1.csv")
n_nonhuman = int((~((raw["assay_tax_id"] == 9606) | (raw["assay_organism"] == "Homo sapiens"))).sum())
n_assaytype = int((~raw["activity_type"].isin({"IC50", "Ki", "Kd", "EC50"})).sum())
n_confidence = int((raw["confidence_score"].fillna(0) < 7).sum())

ledger_rows = [
    ("chembl_actives_n", 31),
    ("chembl_inactives_n", 148),
    ("chembl_ambiguous_excluded_n", 108),
    ("chembl_nonhuman_excluded_n", n_nonhuman),
    ("chembl_assaytype_excluded_n", n_assaytype),
    ("chembl_confidence_excluded_n", n_confidence),
    ("chembl_activity_conflict_excluded_n", 0),
    ("scaffold_groups_train", groups_tr),
    ("scaffold_groups_val", groups_val),
    ("scaffold_groups_test", groups_te),
    ("largest_scaffold_group_train", max_tr),
    ("largest_scaffold_group_val", max_val),
    ("largest_scaffold_group_test", max_te),
    ("split_repairs_n", 0),
    ("cv_folds_n", 5),
    ("inchikey_leakage_removed_n", 0),
]

# 4. Write to results/diagnostics.csv
diag_path = Path("../results/diagnostics.csv")
header = "section,row_id,metric,split,value,rank,compound_id,compound_name,cultivar_source,source_publication,source_attempted,structure_status,reason,final_score,ligand_only_score,ligand_only_rank,failed_rules,pattern,pair_overlap,sensitivity_label,N_star,U_star,notes\n"

# Create file with header if it doesn't exist
if not diag_path.exists() or diag_path.stat().st_size == 0:
    diag_path.write_text(header)

# Append ledger rows
with open(diag_path, "a") as f:
    for i, (metric, value) in enumerate(ledger_rows):
        # section, row_id, metric, split, value, ... empty rest
        row = f"data_feasibility_ledger,{i+1},{metric},all,{value},,,,,,,,,,,,,,,\n"
        f.write(row)

print(f"Wrote {len(ledger_rows)} ledger rows to {diag_path}")
print(pd.read_csv(diag_path).head(10).to_string())

Wrote 16 ledger rows to ..\results\diagnostics.csv
                   section  row_id                               metric       split  value  rank  compound_id  compound_name  cultivar_source  source_publication  source_attempted  structure_status  reason  final_score  ligand_only_score  ligand_only_rank  failed_rules  pattern  pair_overlap  sensitivity_label  N_star  U_star  notes
0  data_feasibility_ledger       1                     chembl_actives_n         all     31   NaN          NaN            NaN              NaN                 NaN               NaN               NaN     NaN          NaN                NaN               NaN           NaN      NaN           NaN                NaN     NaN     NaN    NaN
1  data_feasibility_ledger       2                   chembl_inactives_n         all    148   NaN          NaN            NaN              NaN                 NaN               NaN               NaN     NaN          NaN                NaN               NaN           NaN      NaN 

In [33]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# Define the exact schema from your corrected data.md
cols = [
    "section", "row_id", "metric", "split", "value", "rank", "compound_id",
    "compound_name", "cultivar_source", "source_publication", "source_attempted",
    "structure_status", "reason", "final_score", "ligand_only_score",
    "ligand_only_rank", "failed_rules", "pattern", "pair_overlap",
    "sensitivity_label", "N_star", "U_star", "notes"
]

# Rebuild the data using the counts you just generated
ledger_data = [
    {"section": "data_feasibility_ledger", "row_id": 1, "metric": "chembl_actives_n", "split": "all", "value": 31},
    {"section": "data_feasibility_ledger", "row_id": 2, "metric": "chembl_inactives_n", "split": "all", "value": 148},
    {"section": "data_feasibility_ledger", "row_id": 3, "metric": "chembl_ambiguous_excluded_n", "split": "all", "value": 108},
    {"section": "data_feasibility_ledger", "row_id": 4, "metric": "chembl_nonhuman_excluded_n", "split": "all", "value": 241},
    {"section": "data_feasibility_ledger", "row_id": 5, "metric": "chembl_assaytype_excluded_n", "split": "all", "value": 264},
    {"section": "data_feasibility_ledger", "row_id": 6, "metric": "chembl_confidence_excluded_n", "split": "all", "value": 0},
    {"section": "data_feasibility_ledger", "row_id": 7, "metric": "chembl_activity_conflict_excluded_n", "split": "all", "value": 0},
    {"section": "data_feasibility_ledger", "row_id": 8, "metric": "scaffold_groups_train", "split": "train", "value": 17},
    {"section": "data_feasibility_ledger", "row_id": 9, "metric": "scaffold_groups_val", "split": "validation", "value": 20},
    {"section": "data_feasibility_ledger", "row_id": 10, "metric": "scaffold_groups_test", "split": "test", "value": 44},
    {"section": "data_feasibility_ledger", "row_id": 11, "metric": "largest_scaffold_group_train", "split": "train", "value": 34},
    # Note: I am inferring max_val and max_te from the output you shared; adjust if needed.
    {"section": "data_feasibility_ledger", "row_id": 12, "metric": "largest_scaffold_group_val", "split": "validation", "value": 2},
    {"section": "data_feasibility_ledger", "row_id": 13, "metric": "largest_scaffold_group_test", "split": "test", "value": 1},
    {"section": "data_feasibility_ledger", "row_id": 14, "metric": "split_repairs_n", "split": "all", "value": 0},
    {"section": "data_feasibility_ledger", "row_id": 15, "metric": "cv_folds_n", "split": "all", "value": 5},
    {"section": "data_feasibility_ledger", "row_id": 16, "metric": "inchikey_leakage_removed_n", "split": "all", "value": 0},
]

df_ledger = pd.DataFrame(ledger_data, columns=cols)

diag_path = Path("../results/diagnostics.csv")

# If the file exists, read it, drop old ledger rows, and append the new aligned ones
if diag_path.exists() and diag_path.stat().st_size > 0:
    df_existing = pd.read_csv(diag_path)
    # Keep only rows that aren't ledger rows (e.g., if you already added missing_structures)
    df_existing = df_existing[df_existing["section"] != "data_feasibility_ledger"]
    df_final = pd.concat([df_existing, df_ledger], ignore_index=True)
else:
    df_final = df_ledger

# Ensure all columns exist and are in the right order
df_final = df_final.reindex(columns=cols)
df_final.to_csv(diag_path, index=False)

print(f"Cleaned and wrote {len(df_ledger)} ledger rows.")
display(df_final.head(5))

Cleaned and wrote 16 ledger rows.


,section,row_id,metric,split,value,rank,compound_id,compound_name,cultivar_source,source_publication,...,final_score,ligand_only_score,ligand_only_rank,failed_rules,pattern,pair_overlap,sensitivity_label,N_star,U_star,notes
0,data_feasibility_ledger,1,chembl_actives_n,all,31,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,data_feasibility_ledger,2,chembl_inactives_n,all,148,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,data_feasibility_ledger,3,chembl_ambiguous_excluded_n,all,108,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,data_feasibility_ledger,4,chembl_nonhuman_excluded_n,all,241,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,data_feasibility_ledger,5,chembl_assaytype_excluded_n,all,264,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
import pandas as pd
from rdkit import Chem
from pathlib import Path
from IPython.display import display

# 1. Load P3's catalogue
cat = pd.read_csv("../data/cultivar_compound_catalogue.csv")

# 2. Validate P3's SMILES and InChIKeys
def validate_structure(row):
    mol = Chem.MolFromSmiles(row["canonical_smiles"])
    if mol is None:
        return "invalid_smiles"
    # Check if InChIKey matches the SMILES
    generated_key = Chem.MolToInchiKey(mol)
    if generated_key != row["inchikey"]:
        return "inchikey_mismatch"
    return "valid"

cat["validation_status"] = cat.apply(validate_structure, axis=1)
invalid_rows = cat[cat["validation_status"] != "valid"]
if len(invalid_rows) > 0:
    print("WARNING: Structure validation failed for some rows:")
    display(invalid_rows[["compound_name", "validation_status"]])
else:
    print("All structures validated successfully.")

# 3. Calculate overlaps and counts
# Get P3's unique InChIKeys
cat_keys = set(cat["inchikey"].dropna())
# Get ChEMBL active InChIKeys
chembl_active_keys = set(curated_final[curated_final["activity_class"] == "active"]["standard_inchikey"])
overlap_keys = cat_keys.intersection(chembl_active_keys)

print(f"Unique compounds in catalogue: {cat['compound_id'].nunique()}")
print(f"Unique compounds overlapping with ChEMBL actives: {len(overlap_keys)}")
if len(overlap_keys) > 0:
    print("Overlapping compounds:")
    display(cat[cat["inchikey"].isin(overlap_keys)][["compound_name", "inchikey"]].drop_duplicates())

# 4. Build Provenance Table
prov = cat.copy()
prov["literature_evidence_level"] = prov["inchikey"].apply(
    lambda k: "experimental" if k in overlap_keys else "none"
)
prov = prov[[
    "compound_id", "compound_name", "cultivar_source", "cultivar_evidence_level",
    "literature_evidence_level", "source_publication", "source_locator",
    "structure_status", "structure_source", "structure_source_record_id", "inchikey"
]]
prov.to_csv("../provenance/provenance_table.csv", index=False)
print(f"Wrote {len(prov)} rows to provenance_table.csv")

# 5. Append remaining ledger metrics
new_ledger_data = [
    {"section": "data_feasibility_ledger", "row_id": 17, "metric": "pair_identified_n", "split": "all", "value": len(cat[cat["cultivar_evidence_level"] == "identified"])},
    {"section": "data_feasibility_ledger", "row_id": 18, "metric": "pair_suspected_n", "split": "all", "value": len(cat[cat["cultivar_evidence_level"] == "suspected"])},
    {"section": "data_feasibility_ledger", "row_id": 19, "metric": "structure_defined_n", "split": "all", "value": cat['compound_id'].nunique()},
    {"section": "data_feasibility_ledger", "row_id": 20, "metric": "structure_ambiguous_n", "split": "all", "value": 0},
    {"section": "data_feasibility_ledger", "row_id": 21, "metric": "structure_missing_n", "split": "all", "value": 0},
    {"section": "data_feasibility_ledger", "row_id": 22, "metric": "structure_source_pubchem_n", "split": "all", "value": len(cat[cat["structure_source"] == "PubChem"])},
    {"section": "data_feasibility_ledger", "row_id": 23, "metric": "structure_source_chembl_n", "split": "all", "value": len(cat[cat["structure_source"] == "ChEMBL"])},
    {"section": "data_feasibility_ledger", "row_id": 24, "metric": "structure_exclusion_n", "split": "all", "value": 0},
    {"section": "data_feasibility_ledger", "row_id": 25, "metric": "known_active_overlap_n", "split": "all", "value": len(overlap_keys)},
    {"section": "data_feasibility_ledger", "row_id": 26, "metric": "eligible_rosemary_compound_n", "split": "all", "value": cat['compound_id'].nunique()},
]

df_new_ledger = pd.DataFrame(new_ledger_data)

# Load existing diagnostics to append safely
diag_path = Path("../results/diagnostics.csv")
if diag_path.exists() and diag_path.stat().st_size > 0:
    df_existing = pd.read_csv(diag_path)
    # Drop old versions of these metrics if they exist
    df_existing = df_existing[~df_existing["metric"].isin(df_new_ledger["metric"])]
    df_final = pd.concat([df_existing, df_new_ledger], ignore_index=True)
else:
    df_final = df_new_ledger

df_final.to_csv(diag_path, index=False)
print(f"Successfully updated ledger in diagnostics.csv")

All structures validated successfully.
Unique compounds in catalogue: 28
Unique compounds overlapping with ChEMBL actives: 0
Wrote 64 rows to provenance_table.csv
Successfully updated ledger in diagnostics.csv
